In [ ]:
"""
=============================================================================
NASA CMAPSS FD001 — Data Enhancement Pipeline
=============================================================================
Purpose : Transform the raw FD001 train/test/RUL files into a rich dataset
          ready for:
            • MTTR  (Mean Time To Repair)
            • MTBF  (Mean Time Between Failures)
            • MTU   (Mean Time to Unplanned failure / availability metric)
            • Fault-type classification (bearing misalignment, overheating, wear)

Strategy (per engine):
  1. Split the engine's run into 2–4 segments → each segment is one "cycle"
     between repairs, i.e. one MTBF interval.
  2. The end of every segment = a failure event.
  3. A defect type is assigned via sensor-based heuristics.
  4. A repair duration (downtime) is sampled according to defect type.
  5. Downtime rows are inserted between segments.

Outputs (written to  ./enhanced_data/ ):
  • enriched_timeseries.csv   – original sensor columns + state / defect labels
  • event_log.csv             – one row per failure or repair event
  • data_card.txt             – human-readable data card

Dependencies : pandas, numpy  (standard Colab / pip install)
=============================================================================
"""

# ── 0. Imports ────────────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd

# ── 1. Configuration (edit paths / random seed here) ─────────────────────────
DATA_DIR    = "data"          # folder that contains the raw FD001 files
OUTPUT_DIR  = "enhanced_data"
RANDOM_SEED = 42

# Raw file names (standard CMAPSS naming)
TRAIN_FILE  = os.path.join(DATA_DIR, "train_FD001.txt")
TEST_FILE   = os.path.join(DATA_DIR, "test_FD001.txt")
RUL_FILE    = os.path.join(DATA_DIR, "RUL_FD001.txt")

os.makedirs(OUTPUT_DIR, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
# ── 2. Column schema ──────────────────────────────────────────────────────────
COLS = (
    ["engine_id", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}"     for i in range(1, 22)]
)

# The three sensors we use for defect classification (CMAPSS indices 1-based)
# sensor_2  = HPC outlet temperature  → overheating proxy
# sensor_14 = HPC outlet pressure ratio deviation → bearing misalignment proxy
# sensor_11 = bypass ratio            → wear / degradation proxy
SENSOR_TEMP   = "sensor_2"
SENSOR_BEARING= "sensor_14"
SENSOR_WEAR   = "sensor_11"

In [ ]:
# ── 3. Load raw data ──────────────────────────────────────────────────────────
def load_raw(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None, names=COLS)
    return df

print("Loading raw FD001 files …")
train_raw = load_raw(TRAIN_FILE)
test_raw  = load_raw(TEST_FILE)
rul_raw   = pd.read_csv(RUL_FILE, header=None, names=["true_rul"])

print(f"  Train : {train_raw.shape[0]:,} rows  |  {train_raw['engine_id'].nunique()} engines")
print(f"  Test  : {test_raw.shape[0]:,} rows   |  {test_raw['engine_id'].nunique()} engines")

Loading raw FD001 files …
  Train : 20,631 rows  |  100 engines
  Test  : 13,096 rows   |  100 engines


In [ ]:
# ── 4. Compute sensor percentile bounds (fit on TRAIN only) ──────────────────
# These thresholds drive defect classification; they are data-driven, not magic
# numbers, so the logic stays valid if you swap FD002/003/004.
def compute_thresholds(df: pd.DataFrame, q_low: float = 0.75, q_high: float = 0.90):
    """Return (low, high) percentile for each classification sensor."""
    th = {}
    for col in [SENSOR_TEMP, SENSOR_BEARING, SENSOR_WEAR]:
        th[col] = (df[col].quantile(q_low), df[col].quantile(q_high))
    return th

THRESHOLDS = compute_thresholds(train_raw)
print("\nSensor thresholds (75th / 90th pct on train):")
for col, (lo, hi) in THRESHOLDS.items():
    print(f"  {col:12s}  low={lo:.3f}  high={hi:.3f}")


Sensor thresholds (75th / 90th pct on train):
  sensor_2      low=643.000  high=643.360
  sensor_14     low=8148.310  high=8162.750
  sensor_11     low=47.700  high=47.930


In [ ]:
# ── 5. Defect classification ──────────────────────────────────────────────────
DEFECT_PRIORITY = ["overheating", "bearing_misalignment", "wear", "unclassified"]

# Mean repair durations in cycles (will be used for MTTR)
REPAIR_DURATION = {
    "overheating"        : {"mean": 8,  "std": 2},
    "bearing_misalignment": {"mean": 12, "std": 3},
    "wear"               : {"mean": 20, "std": 5},
    "unclassified"       : {"mean": 6,  "std": 2},
}

def classify_defect(segment: pd.DataFrame, thresholds: dict) -> str:
    """
    Rule-based defect assignment on the LAST 20 % of a segment
    (where degradation signal is strongest).

    Priority  Rule
    --------  ----
    1  Overheating        : mean SENSOR_TEMP  > 90th-pct threshold
    2  Bearing misalign.  : mean SENSOR_BEARING > 90th-pct threshold
    3  Wear               : mean SENSOR_WEAR   > 75th-pct threshold
    4  Unclassified       : none of the above
    """
    tail = segment.tail(max(1, int(len(segment) * 0.20)))

    temp_th_lo, temp_th_hi   = thresholds[SENSOR_TEMP]
    bear_th_lo, bear_th_hi   = thresholds[SENSOR_BEARING]
    wear_th_lo, _            = thresholds[SENSOR_WEAR]

    if tail[SENSOR_TEMP].mean()    > temp_th_hi:
        return "overheating"
    if tail[SENSOR_BEARING].mean() > bear_th_hi:
        return "bearing_misalignment"
    if tail[SENSOR_WEAR].mean()    > wear_th_lo:
        return "wear"
    return "unclassified"

In [ ]:
# ── 6. Core enhancement function ──────────────────────────────────────────────
def enhance_engine(engine_df: pd.DataFrame,
                   engine_id: int,
                   rng: np.random.Generator,
                   thresholds: dict) -> tuple[pd.DataFrame, list[dict]]:
    """
    Splits one engine trajectory into 2–4 failure segments and inserts
    downtime rows between them.

    Returns
    -------
    enriched_rows : pd.DataFrame  – sensor rows tagged with state/defect
    events        : list[dict]    – failure + repair event records
    """
    engine_df = engine_df.reset_index(drop=True)
    total_cycles = len(engine_df)

    # ── 6a. Decide number of segments ────────────────────────────────────────
    # Shorter trajectories → fewer splits (avoid 1-row segments)
    if total_cycles < 60:
        n_seg = 2
    elif total_cycles < 120:
        n_seg = rng.integers(2, 4)      # 2 or 3
    else:
        n_seg = rng.integers(2, 5)      # 2 to 4

    # ── 6b. Choose split points (proportional random cuts) ───────────────────
    cuts = sorted(rng.choice(range(10, total_cycles - 10), size=n_seg - 1, replace=False))
    segment_boundaries = [0] + list(cuts) + [total_cycles]

    enriched_rows = []
    events        = []
    virtual_time  = 0   # monotonically increasing "calendar" cycle counter

    for seg_idx in range(n_seg):
        start = segment_boundaries[seg_idx]
        end   = segment_boundaries[seg_idx + 1]
        segment = engine_df.iloc[start:end].copy()

        seg_len = len(segment)

        # ── 6c. Classify defect ──────────────────────────────────────────────
        defect = classify_defect(segment, thresholds)

        # ── 6d. Tag operational rows ─────────────────────────────────────────
        segment["virtual_cycle"]  = virtual_time + np.arange(seg_len)
        segment["segment_id"]     = seg_idx + 1
        segment["machine_state"]  = "running"
        segment["defect_type"]    = defect          # label assigned at end
        segment["is_failure_end"] = False
        segment.iloc[-1, segment.columns.get_loc("is_failure_end")] = True

        # RUL within segment (counts down to 0 at end)
        segment["segment_rul"] = np.arange(seg_len - 1, -1, -1)

        enriched_rows.append(segment)

        # ── 6e. Failure event ─────────────────────────────────────────────────
        failure_cycle = virtual_time + seg_len - 1
        events.append({
            "engine_id"    : engine_id,
            "segment_id"   : seg_idx + 1,
            "event_type"   : "failure",
            "defect_type"  : defect,
            "virtual_cycle": failure_cycle,
            "duration_cycles": None,
        })
        virtual_time += seg_len

        # ── 6f. Repair / downtime (not after the very last segment) ──────────
        if seg_idx < n_seg - 1:
            repair_params = REPAIR_DURATION[defect]
            repair_dur    = max(1, int(rng.normal(repair_params["mean"],
                                                   repair_params["std"])))

            # Build downtime rows (NaN sensors — machine is offline)
            dt_cycles = virtual_time + np.arange(repair_dur)
            dt_rows   = pd.DataFrame({
                col: np.nan for col in engine_df.columns
            }, index=range(repair_dur))
            dt_rows["engine_id"]      = engine_id
            dt_rows["cycle"]          = np.nan     # no sensor cycle during repair
            dt_rows["virtual_cycle"]  = dt_cycles
            dt_rows["segment_id"]     = seg_idx + 1
            dt_rows["machine_state"]  = "repair"
            dt_rows["defect_type"]    = defect
            dt_rows["is_failure_end"] = False
            dt_rows["segment_rul"]    = np.nan

            enriched_rows.append(dt_rows)

            events.append({
                "engine_id"    : engine_id,
                "segment_id"   : seg_idx + 1,
                "event_type"   : "repair",
                "defect_type"  : defect,
                "virtual_cycle": virtual_time,
                "duration_cycles": repair_dur,
            })
            virtual_time += repair_dur

    enriched = pd.concat(enriched_rows, ignore_index=True)
    return enriched, events

In [ ]:
# ── 7. Apply to all engines in the TRAIN set ─────────────────────────────────
print("\nEnhancing training engines …")

all_enriched = []
all_events   = []

for eid, grp in train_raw.groupby("engine_id"):
    enriched, events = enhance_engine(grp, eid, rng, THRESHOLDS)
    all_enriched.append(enriched)
    all_events.extend(events)
    if eid % 20 == 0:
        print(f"  … processed engine {eid}")

enriched_df = pd.concat(all_enriched, ignore_index=True)
event_log   = pd.DataFrame(all_events)

print(f"\nDone.  Enriched rows : {len(enriched_df):,}")
print(f"       Event records : {len(event_log):,}")


Enhancing training engines …
  … processed engine 20
  … processed engine 40
  … processed engine 60
  … processed engine 80
  … processed engine 100

Done.  Enriched rows : 22,161
       Event records : 496


In [ ]:
# ── 8. Derive KPI summary columns in the event log ───────────────────────────
# MTBF = mean operational duration between failures (per engine)
# MTTR = mean repair duration (per engine)
# MTU  = Mean Time to Unplanned failure = same as MTBF here (all failures
#         are treated as unplanned since the engine ran to end of segment)

print("\nComputing per-engine KPIs …")

def compute_kpis(event_log: pd.DataFrame, enriched_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for eid in event_log["engine_id"].unique():
        eng_events = event_log[event_log["engine_id"] == eid].copy()
        failures   = eng_events[eng_events["event_type"] == "failure"]
        repairs    = eng_events[eng_events["event_type"] == "repair"]

        # Operational time per segment = length of running rows
        eng_ts     = enriched_df[
            (enriched_df["engine_id"] == eid) &
            (enriched_df["machine_state"] == "running")
        ]
        op_times   = eng_ts.groupby("segment_id").size()

        mtbf = op_times.mean()  if len(op_times) > 0 else np.nan
        mtu  = mtbf                                    # all failures unplanned

        if len(repairs) > 0:
            mttr = repairs["duration_cycles"].mean()
        else:
            mttr = 0.0

        # Availability = uptime / (uptime + downtime)
        total_uptime   = op_times.sum()
        total_downtime = repairs["duration_cycles"].sum() if len(repairs) > 0 else 0
        availability   = total_uptime / (total_uptime + total_downtime) if (total_uptime + total_downtime) > 0 else np.nan

        rows.append({
            "engine_id"       : eid,
            "n_failure_cycles": len(failures),
            "MTBF_cycles"     : round(mtbf, 2),
            "MTTR_cycles"     : round(mttr, 2),
            "MTU_cycles"      : round(mtu,  2),
            "availability_pct": round(availability * 100, 2),
        })

    return pd.DataFrame(rows)

kpi_df = compute_kpis(event_log, enriched_df)
print(kpi_df.describe().round(2))


Computing per-engine KPIs …
       engine_id  n_failure_cycles  MTBF_cycles  MTTR_cycles  MTU_cycles  \
count     100.00            100.00       100.00       100.00      100.00   
mean       50.50              2.98        75.68         7.40       75.68   
std        29.01              0.85        29.29         3.63       29.29   
min         1.00              2.00        32.00         1.00       32.00   
25%        25.75              2.00        52.00         5.00       52.00   
50%        50.50              3.00        68.88         7.00       68.88   
75%        75.25              4.00        94.42         9.00       94.42   
max       100.00              4.00       181.00        19.33      181.00   

       availability_pct  
count            100.00  
mean              93.06  
std                4.90  
min               75.96  
25%               91.21  
50%               94.36  
75%               96.61  
max               99.57  


In [ ]:
# ── 9. Save outputs ───────────────────────────────────────────────────────────
ts_path    = os.path.join(OUTPUT_DIR, "enriched_timeseries.csv")
event_path = os.path.join(OUTPUT_DIR, "event_log.csv")
kpi_path   = os.path.join(OUTPUT_DIR, "kpi_summary.csv")

enriched_df.to_csv(ts_path,    index=False)
event_log.to_csv(  event_path, index=False)
kpi_df.to_csv(     kpi_path,   index=False)

print(f"\nFiles written to '{OUTPUT_DIR}/':")
print(f"  {ts_path}")
print(f"  {event_path}")
print(f"  {kpi_path}")


Files written to 'enhanced_data/':
  enhanced_data/enriched_timeseries.csv
  enhanced_data/event_log.csv
  enhanced_data/kpi_summary.csv


In [ ]:
# ── 10. Data Card ─────────────────────────────────────────────────────────────
data_card = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║              DATA CARD — Enhanced NASA CMAPSS FD001 Dataset                ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SOURCE DATASET
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Name        : NASA CMAPSS (Commercial Modular Aero-Propulsion System
               Simulation) — Subset FD001
 Source      : NASA Ames Prognostics Data Repository
 Citation    : Saxena et al. (2008), PHM08 challenge data
 Condition   : One operating condition, one fault mode (HPC degradation)
 Engines     : {train_raw['engine_id'].nunique()} training engines
 Raw rows    : {len(train_raw):,} (train)
 Raw columns : 26  (engine_id, cycle, 3 op-settings, 21 sensors)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 ENHANCEMENT METHODOLOGY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Step 1 — Trajectory Segmentation
   Each engine's run (e.g. 200 cycles) is split into 2–4 non-overlapping
   segments at randomly chosen cut points (min 10 cycles from either end).
   Each segment represents one failure-to-repair cycle (MTBF interval).
   Segments are shorter for shorter engines to avoid degenerate 1-row slices.

 Step 2 — Failure Labelling
   The last row of every segment is flagged as is_failure_end = True.
   All segments end in failure (run-to-failure assumption, matching the
   original CMAPSS design intent).

 Step 3 — Defect Classification  (sensor-rule heuristics)
   Classification is applied to the LAST 20 % of each segment
   (the region where degradation signal is richest).

   Sensor proxy mapping:
     sensor_2  → HPC outlet temperature  → OVERHEATING
     sensor_14 → Pressure ratio deviation → BEARING MISALIGNMENT
     sensor_11 → Bypass ratio             → WEAR

   Priority rules (first match wins):
     1. mean(sensor_2)  > 90th-pct threshold  → overheating
     2. mean(sensor_14) > 90th-pct threshold  → bearing_misalignment
     3. mean(sensor_11) > 75th-pct threshold  → wear
     4. otherwise                             → unclassified

   Thresholds are computed from the TRAINING set only to prevent
   data leakage.

 Step 4 — Repair Duration Simulation
   Repair durations are sampled from Normal distributions fitted to
   domain-inspired means and standard deviations:

     Defect type           Mean (cycles)   Std
     ─────────────────── ─────────────── ─────
     overheating                 8          2
     bearing_misalignment       12          3
     wear                       20          5
     unclassified                6          2

   Duration is clipped to minimum 1 cycle.

 Step 5 — Downtime Insertion
   Between consecutive segments, 'repair' rows are inserted.
   These rows have NaN sensor values (machine offline).
   A monotonic virtual_cycle counter tracks wall-clock time across
   segments and downtimes, enabling MTBF / MTTR computation.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 OUTPUT FILES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 1. enriched_timeseries.csv   ({len(enriched_df):,} rows)
    ─────────────────────────
    All original columns PLUS:

    Column           Type     Description
    ───────────────  ───────  ──────────────────────────────────────────────
    virtual_cycle    int      Monotonic wall-clock cycle (includes downtime)
    segment_id       int      Which segment (1 = first failure cycle)
    machine_state    str      'running' | 'repair'
    defect_type      str      'overheating' | 'bearing_misalignment' |
                              'wear' | 'unclassified'
    is_failure_end   bool     True only on the failure row of each segment
    segment_rul      float    Cycles remaining until end of THIS segment
                              (NaN during repair rows)

 2. event_log.csv   ({len(event_log):,} rows)
    ───────────────
    One row per event (failure or repair).

    Column           Type     Description
    ───────────────  ───────  ──────────────────────────────────────────────
    engine_id        int      Engine identifier
    segment_id       int      Segment that generated this event
    event_type       str      'failure' | 'repair'
    defect_type      str      Defect that caused the failure / repair
    virtual_cycle    int      Wall-clock cycle when event started
    duration_cycles  float    Repair duration (NaN for failure events)

 3. kpi_summary.csv   ({len(kpi_df):,} rows — one per engine)
    ────────────────
    Column              Description
    ──────────────────  ────────────────────────────────────────────────────
    engine_id           Engine identifier
    n_failure_cycles    Number of failure events for this engine
    MTBF_cycles         Mean Time Between Failures (mean operational segment
                        length in virtual cycles)
    MTTR_cycles         Mean Time To Repair (mean repair duration in cycles)
    MTU_cycles          Mean Time to Unplanned failure  ← equals MTBF here
                        because all failures are unplanned (run-to-failure)
    availability_pct    Uptime / (Uptime + Downtime) × 100

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 KPI DEFINITIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 MTBF (Mean Time Between Failures)
   Average number of operational cycles an engine runs before a failure.
   Computed per engine as the mean segment length across all segments.
   Formula: MTBF = Σ operational_cycles_per_segment / n_segments

 MTTR (Mean Time To Repair)
   Average number of cycles spent repairing after a failure.
   Computed per engine as the mean repair duration across all repair events.
   Formula: MTTR = Σ repair_duration_per_event / n_repair_events

 MTU (Mean Time to Unplanned failure)
   In this dataset all failures are unplanned (engines run to failure).
   Therefore MTU = MTBF.
   In a real deployment with scheduled maintenance, only unplanned failures
   would count, making MTU ≥ MTBF.

 Availability
   Proportion of calendar time the engine is operational.
   Formula: A = MTBF / (MTBF + MTTR)   [expressed as %]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SUMMARY STATISTICS (training engines)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Enriched rows         : {len(enriched_df):,}
 Running rows          : {(enriched_df['machine_state']=='running').sum():,}
 Repair (downtime) rows: {(enriched_df['machine_state']=='repair').sum():,}
 Failure events        : {(event_log['event_type']=='failure').sum():,}
 Repair events         : {(event_log['event_type']=='repair').sum():,}

 Defect distribution (failure events):
{event_log[event_log['event_type']=='failure']['defect_type'].value_counts().to_string()}

 Fleet-level KPIs:
   Mean MTBF        : {kpi_df['MTBF_cycles'].mean():.1f} cycles
   Mean MTTR        : {kpi_df['MTTR_cycles'].mean():.1f} cycles
   Mean Availability: {kpi_df['availability_pct'].mean():.1f} %

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 LIMITATIONS & CAVEATS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 • Repair durations are SIMULATED, not measured. They reflect assumed
   domain knowledge and should be replaced with real maintenance logs
   if available.
 • Defect labels are RULE-BASED proxies. The original CMAPSS dataset has
   only one real fault mode (HPC degradation). The three defect classes
   created here are soft mappings via sensor correlates — not ground truth.
 • Segment boundaries are randomly chosen. Real maintenance intervals would
   follow planned schedules or alarm thresholds.
 • The test set is NOT enhanced here because true failure cycles are unknown
   (only the RUL at cut-off is given). A separate inference step would be
   needed to project test engines into this framework.
 • "virtual_cycle" is an artificial clock. It is consistent within one
   engine but should NOT be compared across engines without normalisation.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 RANDOM SEED : {RANDOM_SEED}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

card_path = os.path.join(OUTPUT_DIR, "data_card.txt")
with open(card_path, "w", encoding="utf-8") as f:
    f.write(data_card)

print(data_card)
print(f"Data card saved to '{card_path}'")


╔══════════════════════════════════════════════════════════════════════════════╗
║              DATA CARD — Enhanced NASA CMAPSS FD001 Dataset                ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SOURCE DATASET
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Name        : NASA CMAPSS (Commercial Modular Aero-Propulsion System
               Simulation) — Subset FD001
 Source      : NASA Ames Prognostics Data Repository
 Citation    : Saxena et al. (2008), PHM08 challenge data
 Condition   : One operating condition, one fault mode (HPC degradation)
 Engines     : 100 training engines
 Raw rows    : 20,631 (train)
 Raw columns : 26  (engine_id, cycle, 3 op-settings, 21 sensors)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 ENHANCEMENT METHODOLOGY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ── 11. Quick sanity checks ───────────────────────────────────────────────────
print("\n── Sanity checks ──────────────────────────────────────────────────────")

# Every failure should have a following repair (except the very last segment)
failures_per_engine = event_log[event_log["event_type"] == "failure"].groupby("engine_id").size()
repairs_per_engine  = event_log[event_log["event_type"] == "repair" ].groupby("engine_id").size()
expected_repairs    = failures_per_engine - 1   # last segment has no repair
mismatch            = (repairs_per_engine != expected_repairs).sum()
print(f"  Engines with repair-count mismatch : {mismatch}  (expected 0)")

# No running rows should have NaN sensor values
running_mask  = enriched_df["machine_state"] == "running"
nan_in_running = enriched_df.loc[running_mask, "sensor_1"].isna().sum()
print(f"  Running rows with NaN sensor_1     : {nan_in_running}  (expected 0)")

# All repair rows should have NaN sensor values
repair_mask   = enriched_df["machine_state"] == "repair"
not_nan_repair = enriched_df.loc[repair_mask, "sensor_1"].notna().sum()
print(f"  Repair rows with non-NaN sensor_1  : {not_nan_repair}  (expected 0)")

print("\n✓ Pipeline complete. All outputs in ./enhanced_data/")


── Sanity checks ──────────────────────────────────────────────────────
  Engines with repair-count mismatch : 0  (expected 0)
  Running rows with NaN sensor_1     : 0  (expected 0)
  Repair rows with non-NaN sensor_1  : 0  (expected 0)

✓ Pipeline complete. All outputs in ./enhanced_data/
